In [6]:
import xarray as xr
import numpy as np

import sys
sys.path.append('../new_model')
from physics_equations import specific_humidity_from_rh
from constants import g

In [7]:
# calculating the surface specific humidity and inserting it to the profiles
def relative_humidity(t_air, t_dew):
    """
    t_air and t_dew in Celsius
    Returns RH in %
    """
    rh = 100 * (np.exp((17.67 * t_dew) / (t_dew + 243.5)) / 
                np.exp((17.67 * t_air) / (t_air + 243.5)))
    return rh



In [8]:
#profile data
data_era = xr.open_mfdataset(r'..\data\ERA5\profiles\*.nc')
data_era = data_era.sel(latitude=51.96,longitude=4.9,method='nearest')
data_era = data_era.sortby('pressure_level', ascending=False)
data_era = data_era.sel(pressure_level=slice(1e5,200))


my_temp = data_era.t.values
p = data_era.pressure_level*100
q_v = data_era.q.values
Z = data_era.z.values/g     #z is geopotential in ERA5

#surface data
data_surf_era = xr.open_mfdataset(r'..\data\ERA5\surface\*.nc')
data_surf_era = data_surf_era.sel(latitude=51.96,longitude=4.9,method='nearest')

my_temp_surface = data_surf_era.t2m.values
p_surface = data_surf_era.sp.values
T_d = data_surf_era.d2m.values
Z_surface = 2*np.ones(len(p_surface))  

#inserting the surface measurements at the profiles
T_air = np.concatenate([my_temp_surface.reshape(len(my_temp_surface), 1), my_temp], 1)
p_surf = p_surface #[Pa]
p_air = np.tile(p.values,(len(p_surf),1))
p_air = np.concatenate([p_surf.reshape(len(p_surf), 1), p_air], 1) #[Pa]
Z = np.concatenate([Z_surface.reshape(len(Z_surface), 1), Z], 1)

RH_surf = relative_humidity(my_temp_surface, T_d)
q_v_surf = specific_humidity_from_rh(my_temp_surface,RH_surf/100, p_surf)
q_v_air = np.concatenate([q_v_surf.reshape(len(q_v_surf), 1), q_v], 1) 

In [9]:
time = data_era.valid_time

# Create xarray Dataset
ds_profiles = xr.Dataset(
    {
        "temperature": (["valid_time", "level"], T_air),
        "pressure": (["valid_time", "level"], p_air),
        "height": (["valid_time", "level"], Z),
        "specific_humidity": (["valid_time", "level"], q_v_air),
        "dew_point_temperature": (["valid_time"], T_d)
    },
    coords={
        "valid_time": time,
        "level": p_air[0, :]
    }
)

# Add metadata
ds_profiles["temperature"].attrs["units"] = "K"
ds_profiles["pressure"].attrs["units"] = "Pa"
ds_profiles["height"].attrs["units"] = "m"
ds_profiles["specific_humidity"].attrs["units"] = "kg/kg"
ds_profiles["dew_point_temperature"].attrs["units"] = "K"

# Save to NetCDF
ds_profiles.to_netcdf(r"..\data\model_inputs\ERA5\ERA5combined_profile_data.nc")